# Production RAG System (Google Gemini & LangChain)
**Author:** Mostafa Ihab  
**Date:** March 2026  
**Version:** 1.1.0  

An end-to-end Retrieval-Augmented Generation (RAG) system utilizing Google Gemini, Chroma DB, and BM25 hybrid retrieval.

In [ ]:
# Cell 1: Install dependencies
!pip install -q langchain langchain-core langchain-community \
    langchain-google-genai langchain-chroma \
    chromadb langsmith \
    pypdf tiktoken sentence-transformers \
    pymupdf docx2txt langchain-experimental unstructured rank_bm25 langchain-classic \
    python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 5.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 24.2 MB/s eta 0:00:00
   ━━━━━

In [ ]:
# Gathering all imports
import os
import re
from pathlib import Path
import hashlib
from datetime import datetime

# Load environment variables from .env file (if available)
try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

from langchain_core.documents import Document
from langchain_community.document_loaders import (
    PyMuPDFLoader, Docx2txtLoader, TextLoader, WebBaseLoader,
    UnstructuredMarkdownLoader, CSVLoader
)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker
from langchain_google_genai import GoogleGenerativeAIEmbeddings

/tmp/ipykernel_1043/1590410118.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (
/tmp/ipykernel_1043/1590410118.py:15: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


In [ ]:
# Cell 2: config.py content

def setup_environment():
    # 1. Load local .env file
    if load_dotenv:
        load_dotenv()

    # 2. Fallback to Google Colab userdata if running in Colab
    try:
        from google.colab import userdata
        if not os.environ.get("GOOGLE_API_KEY") and userdata.get('GOOGLE_API_KEY'):
            os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
        if not os.environ.get("LANGCHAIN_API_KEY") and userdata.get('LANGCHAIN_API_KEY'):
            os.environ["LANGCHAIN_API_KEY"] = userdata.get('LANGCHAIN_API_KEY')
    except ImportError:
        pass

    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "gemini-rag-local")
    os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"

    if not os.environ.get("GOOGLE_API_KEY"):
        print("Warning: GOOGLE_API_KEY is not set. Please add it to your .env file or environment.")
    else:
        print("[INFO] Environment configured. LangSmith project:", os.environ["LANGCHAIN_PROJECT"])

setup_environment()

✅ Environment configured. LangSmith project: gemini-rag-colab


In [ ]:

def get_embedding_model():
    return GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-2")

#DATA Processing

## Data Loader

In [ ]:
# ingestion.py — Part A: Universal loader


LOADER_MAP = {
    ".pdf": PyMuPDFLoader,       # better than PyPDFLoader — preserves layout/tables
    ".docx": Docx2txtLoader,
    ".txt": TextLoader,
    ".md": UnstructuredMarkdownLoader,
    ".csv": CSVLoader,
}

def load_single_file(path: str):
    """Load one file, tagging it with rich metadata. Never raises — returns [] on failure."""
    ext = Path(path).suffix.lower()
    loader_cls = LOADER_MAP.get(ext)
    if loader_cls is None:
        print(f"[WARNING] Skipping unsupported file type: {path}")
        return []

    try:
        loader = loader_cls(path)
        docs = loader.load()
        for d in docs:
            d.metadata.update({
                "source_file": Path(path).name,
                "file_type": ext,
                "ingested_at": datetime.utcnow().isoformat(),
            })
        return docs
    except Exception as e:
        print(f"[ERROR] Failed to load {path}: {e}")
        return []

def load_from_directory(dir_path: str, extensions: list[str] = None):
    """Bulk-load every supported file in a folder (e.g. ./data)."""
    extensions = extensions or list(LOADER_MAP.keys())
    all_docs = []
    files = [f for f in Path(dir_path).rglob("*") if f.suffix.lower() in extensions]
    print(f" Found {len(files)} files to load")

    for f in files:
        docs = load_single_file(str(f))
        all_docs.extend(docs)
        print(f"  - {f.name}: {len(docs)} doc(s)")

    print(f"[INFO] Loaded {len(all_docs)} total documents")
    return all_docs

def load_from_urls(urls: list[str]):
    """Web pages — separate path since WebBaseLoader batches well."""
    loader = WebBaseLoader(urls)
    docs = loader.load()
    for d in docs:
        d.metadata["file_type"] = "web"
        d.metadata["ingested_at"] = datetime.utcnow().isoformat()
    print(f"[INFO] Loaded {len(docs)} web pages")
    return docs

 ## Data Cleaner

In [ ]:
def clean_document_text(doc: Document) -> Document:
    """
    Strips junk characters, normalizes whitespace, and cleans the text
    payload of a LangChain Document.
    """
    text = doc.page_content

    # 1. Remove zero-width characters and unprintable unicode characters
    text = text.replace('\u200b', '').replace('\ufeff', '')

    # 2. NOW collapse consecutive whitespaces, tabs, and newlines
    text = re.sub(r'\s+', ' ', text)

    # 3. Strip leading/trailing spaces
    doc.page_content = text.strip()

    return doc

def process_and_clean_docs(raw_docs: list[Document]) -> list[Document]:
    """Runs the cleaner over a list of documents and checks for empty scans."""
    cleaned_docs = []

    for doc in raw_docs:
        cleaned = clean_document_text(doc)

        # Check if the document is suspiciously empty (likely a scanned image)
        if len(cleaned.page_content) < 10:
            filename = cleaned.metadata.get('source_file', 'Unknown File')
            print(f"Warning: '{filename}' has almost no text. Is it a scanned image requiring OCR?")
            continue # Skip adding empty documents to our database

        cleaned_docs.append(cleaned)

    return cleaned_docs

## Duplicate removal

In [ ]:
def deduplicate_docs(docs):
    seen = set()
    unique = []
    for d in docs:
        h = hashlib.md5(d.page_content.encode()).hexdigest()
        if h not in seen:
            seen.add(h)
            unique.append(d)
    print(f" Deduped {len(docs)} → {len(unique)} documents")
    return unique

## Testing data

In [ ]:
print(" Starting Data Pipeline Test...\n")

# 1. Create a deliberately messy dataset
messy_docs = [
    # Document 1: Poor formatting, hidden Unicode characters, excessive whitespace
    Document(
        page_content="""
        Modern   semiconductor \t\t manufacturing relies on highly precise
        \u200b photolithography, ion implantation, and chemical vapor deposition
        processes.      Silicon wafers are fabricated in cleanroom environments
        where airborne particles must be minimized to prevent defects in integrated
        circuits.


        As transistor dimensions continue to shrink below ten nanometers,
        manufacturers face increasing challenges related to power consumption,
        heat dissipation, and quantum effects.     Advanced process nodes require
        innovations such as FinFET and Gate-All-Around transistor architectures
        to maintain performance improvements while reducing energy usage.
        """,
        metadata={"source_file": "messy_semiconductor_report.pdf", "page": 1}
    ),

    # Document 2: Same content after cleaning (tests deduplication)
    Document(
        page_content="""
        Modern semiconductor manufacturing relies on highly precise photolithography,
        ion implantation, and chemical vapor deposition processes. Silicon wafers
        are fabricated in cleanroom environments where airborne particles must be
        minimized to prevent defects in integrated circuits.

        As transistor dimensions continue to shrink below ten nanometers,
        manufacturers face increasing challenges related to power consumption,
        heat dissipation, and quantum effects. Advanced process nodes require
        innovations such as FinFET and Gate-All-Around transistor architectures
        to maintain performance improvements while reducing energy usage.
        """,
        metadata={"source_file": "duplicate_semiconductor_report.pdf", "page": 1}
    ),

    # Document 3: Simulated scanned or empty PDF page
    Document(
        page_content=" \n \t \u200b \n\n \t ",
        metadata={"source_file": "scanned_datasheet.pdf", "page": 1}
    ),

    # Document 4: Well-formatted document
    Document(
        page_content="""
        Semiconductor devices form the foundation of modern electronic systems,
        including smartphones, medical equipment, automotive control units,
        industrial automation systems, and cloud computing infrastructure.
        Integrated circuits contain millions or even billions of transistors
        connected through multiple metal interconnect layers to perform complex
        computational tasks.

        Continuous advances in materials science, chip packaging, and fabrication
        technologies have enabled higher processing speeds, improved energy
        efficiency, and greater device reliability. Research into compound
        semiconductors such as gallium nitride (GaN) and silicon carbide (SiC)
        is driving the next generation of high-power and high-frequency
        electronic applications.
        """,
        metadata={"source_file": "semiconductor_overview.txt", "page": 1}
    ),

    # Document 5: exactly same as Doc 4
    Document(
        page_content="""
        Semiconductor devices form the foundation of modern electronic systems,
        including smartphones, medical equipment, automotive control units,
        industrial automation systems, and cloud computing infrastructure.
        Integrated circuits contain millions or even billions of transistors
        connected through multiple metal interconnect layers to perform complex
        computational tasks.

        Continuous advances in materials science, chip packaging, and fabrication
        technologies have enabled higher processing speeds, improved energy
        efficiency, and greater device reliability. Research into compound
        semiconductors such as gallium nitride (GaN) and silicon carbide (SiC)
        is driving the next generation of high-power and high-frequency
        electronic applications.
        """,
        metadata={"source_file": "semiconductor_overview(1).txt", "page": 1}
    )
]


print(f" Loaded {len(messy_docs)} raw documents.")

# 2. Run the Cleansing Phase
print("\n Running Data Cleansing Layer...")
cleaned_docs = process_and_clean_docs(messy_docs)

print(f" Survived Cleansing: {len(cleaned_docs)} documents.")
for i, doc in enumerate(cleaned_docs):
    print(f"  [{i}] Source: {doc.metadata['source_file']} | Content: '{doc.page_content}'")

# 3. Run the Deduplication Phase
print("\n️ Running Deduplication Engine...")
final_unique_docs = deduplicate_docs(cleaned_docs)

print(f" Final Production-Ready Documents: {len(final_unique_docs)}")
for i, doc in enumerate(final_unique_docs):
    print(f"  [{i}] Source: {doc.metadata['source_file']} | Content: '{doc.page_content}'")

🧪 Starting Data Pipeline Test...

📥 Loaded 5 raw documents.

🧹 Running Data Cleansing Layer...
⚠️ Warning: 'scanned_datasheet.pdf' has almost no text. Is it a scanned image requiring OCR?
✨ Survived Cleansing: 4 documents.
  [0] Source: messy_semiconductor_report.pdf | Content: 'Modern semiconductor manufacturing relies on highly precise photolithography, ion implantation, and chemical vapor deposition processes. Silicon wafers are fabricated in cleanroom environments where airborne particles must be minimized to prevent defects in integrated circuits. As transistor dimensions continue to shrink below ten nanometers, manufacturers face increasing challenges related to power consumption, heat dissipation, and quantum effects. Advanced process nodes require innovations such as FinFET and Gate-All-Around transistor architectures to maintain performance improvements while reducing energy usage.'
  [1] Source: duplicate_semiconductor_report.pdf | Content: 'Modern semiconductor manufacturing

# Data Chunking

## Recursive Chunking

In [ ]:
class my_RecursiveChunker:
    """Fast, syntax-based chunking using character counts and overlaps."""

    def __init__(self, chunk_size: int = 1000, chunk_overlap: int = 150):
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            length_function=len
        )

    def split(self, documents: list[Document]) -> list[Document]:
        if not documents:
            return []

        print(f"\n Executing RECURSIVE Chunking on {len(documents)} document(s)...")
        chunked_docs = self.splitter.split_documents(documents)
        print(f"[INFO] Generated {len(chunked_docs)} chunks.")

        return chunked_docs

## Semantic Chunking

In [ ]:
class my_SemanticChunker:
    """Advanced, embedding-based chunking that cuts text when the topic shifts."""

    def __init__(self, embedding_model, percentile: int = 80):
        if not embedding_model:
            raise ValueError("[ERROR] You must provide an 'embedding_model'.")

        self.splitter = SemanticChunker(
            embedding_model,
            breakpoint_threshold_type="percentile",
            breakpoint_threshold_amount=percentile
        )

    def split(self, documents: list[Document]) -> list[Document]:
        if not documents:
            return []

        print(f"\n Executing SEMANTIC Chunking on {len(documents)} document(s)...")
        chunked_docs = self.splitter.split_documents(documents)
        print(f"[INFO] Generated {len(chunked_docs)} concept-pure chunks.")

        return chunked_docs

## Testing Chunking

In [ ]:
# 1. Setup the test data
test_text = (
    "The Roman Empire was one of the most powerful empires in history, leaving a massive "
    "legacy on law, language, and architecture. Rome's military legions were highly disciplined. "
    "In completely unrelated news, quantum computing uses qubits to process information in ways "
    "classical computers cannot. By leveraging superposition, they solve complex math instantly."
)
test_docs = [Document(page_content=test_text, metadata={"source": "test_comparison.txt"})]

# 2. Test Recursive
# Set to 500 chars to force it to try and group things
recursive_chunker = my_RecursiveChunker(chunk_size=500, chunk_overlap=0)
recursive_results = recursive_chunker.split(test_docs)

print("\n--- RECURSIVE RESULTS ---")
for i, chunk in enumerate(recursive_results):
    print(f"Chunk {i+1}: '{chunk.page_content}'")

# 3. Test Semantic
embeddings = get_embedding_model()
# Assuming 'embeddings' is your GoogleGenAIEmbeddings instance from earlier
semantic_chunker = my_SemanticChunker(embedding_model=embeddings, percentile=80)
semantic_results = semantic_chunker.split(test_docs)

print("\n--- SEMANTIC RESULTS ---")
for i, chunk in enumerate(semantic_results):
    print(f"Chunk {i+1}: '{chunk.page_content}'")

# Checking metadata preservation on the first output
if semantic_results:
    first_chunk_meta = semantic_results[0].metadata # zero_bracket semantic_results
    print(f"\nMetadata check: {first_chunk_meta}")


🔪 Executing RECURSIVE Chunking on 1 document(s)...
✅ Generated 1 chunks.

--- RECURSIVE RESULTS ---
Chunk 1: 'The Roman Empire was one of the most powerful empires in history, leaving a massive legacy on law, language, and architecture. Rome's military legions were highly disciplined. In completely unrelated news, quantum computing uses qubits to process information in ways classical computers cannot. By leveraging superposition, they solve complex math instantly.'

🧠 Executing SEMANTIC Chunking on 1 document(s)...
✅ Generated 2 concept-pure chunks.

--- SEMANTIC RESULTS ---
Chunk 1: 'The Roman Empire was one of the most powerful empires in history, leaving a massive legacy on law, language, and architecture.'
Chunk 2: 'Rome's military legions were highly disciplined. In completely unrelated news, quantum computing uses qubits to process information in ways classical computers cannot. By leveraging superposition, they solve complex math instantly.'

Metadata check: {'source': 'test_co

# Vector Database

In [ ]:
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_classic.storage import LocalFileStore

def get_cached_embedding_model(cache_dir: str = "./embedding_cache"):
    """
    Wraps get_embedding_model() with a local file-based cache.

    namespace=model name: so if you ever switch embedding models, the cache keys
    don't collide with vectors produced by a different model.
    """
    underlying_embeddings = get_embedding_model()
    store = LocalFileStore(cache_dir)
    return CacheBackedEmbeddings.from_bytes_store(
        underlying_embeddings,
        store,
        namespace=underlying_embeddings.model,
    )


In [ ]:
import time
import hashlib
from langchain_chroma import Chroma

class VectorDBManager:
    """
    Manages the connection to the Chroma vector database and handles
    the safe 'upsert' of documents to prevent duplicates across sessions.
    """
    def __init__(self, collection_name: str = "rag_production", persist_directory: str = "./chroma_db"):
        print(" Initializing Google Gemini Embedding Engine...")
        self.embeddings = get_embedding_model()

        print(f"️ Connecting to local Chroma Database (Dir: {persist_directory})...")
        self.vector_store = Chroma(
            collection_name=collection_name,
            embedding_function=self.embeddings,
            persist_directory=persist_directory
        )

    def _generate_chunk_id(self, chunk: Document) -> str:
        """Generates a deterministic MD5 hash to use as the chunk's database ID."""
        source = chunk.metadata.get('source_file', 'unknown')
        unique_string = f"{chunk.page_content}-{source}"
        return hashlib.md5(unique_string.encode('utf-8')).hexdigest()

    def upsert_chunks(self, chunks: list[Document], batch_size: int = 40):
        """Embeds and saves chunks into Chroma in rate-limited batches with retry backoff."""
        if not chunks:
            print("[WARNING] No chunks provided to database.")
            return

        # 1. Fetch already existing IDs in Chroma
        try:
            existing_data = self.vector_store.get()
            existing_ids = set(existing_data.get("ids", []))
        except Exception:
            existing_ids = set()

        # 2. Filter out already indexed chunks
        chunks_to_add = []
        ids_to_add = []
        for chunk in chunks:
            cid = self._generate_chunk_id(chunk)
            if cid not in existing_ids:
                chunks_to_add.append(chunk)
                ids_to_add.append(cid)

        if not chunks_to_add:
            print(f"[INFO] All {len(chunks)} chunks are already indexed in Chroma. Skipping.")
            return

        already_indexed = len(chunks) - len(chunks_to_add)
        if already_indexed > 0:
            print(f"[INFO] {already_indexed} chunks already present in Chroma index.")
        print(f" Generating vectors and upserting {len(chunks_to_add)} new chunks into Chroma...")

        # 3. Process in batches with 429 backoff
        total_batches = (len(chunks_to_add) + batch_size - 1) // batch_size
        for i in range(0, len(chunks_to_add), batch_size):
            batch_chunks = chunks_to_add[i:i + batch_size]
            batch_ids = ids_to_add[i:i + batch_size]
            batch_num = (i // batch_size) + 1

            for attempt in range(6):
                try:
                    print(f"   -> [Batch {batch_num}/{total_batches}] Embedding {len(batch_chunks)} chunks...")
                    self.vector_store.add_documents(documents=batch_chunks, ids=batch_ids)
                    break
                except Exception as e:
                    err_msg = str(e)
                    if "429" in err_msg or "RESOURCE_EXHAUSTED" in err_msg or "quota" in err_msg.lower():
                        wait_seconds = 30 + (attempt * 10)
                        print(f"    Rate limit hit (429 Resource Exhausted). Waiting {wait_seconds}s before retrying...")
                        time.sleep(wait_seconds)
                    else:
                        raise e

            if batch_num < total_batches:
                time.sleep(1.5)

        print("[INFO] Upsert complete! Your data is now mathematically searchable.")

    def get_retriever(self, top_k: int = 4):
        """Returns the search engine interface for the chatbot."""
        return self.vector_store.as_retriever(search_kwargs={"k": top_k})

In [ ]:
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever

class HybridRetrieverManager:
    """
    Upgrades a standard vector database into a production-grade Hybrid Search engine
    by merging semantic vector search with exact-keyword BM25 search.
    """
    def __init__(self, vector_store, documents, top_k: int = 4):
        print("️ Initializing Hybrid Retrieval Engine...")

        # 1. Setup the Keyword Search Engine (BM25)
        # This builds an in-memory index of exact words and their frequencies
        print("   -> Building BM25 Sparse Keyword Index...")
        self.bm25_retriever = BM25Retriever.from_documents(documents)
        self.bm25_retriever.k = top_k

        # 2. Setup the Semantic Search Engine (Chroma)
        # This uses the vectors we created in Phase 2
        print("   -> Connecting Chroma Dense Vector Index...")
        self.vector_retriever = vector_store.as_retriever(search_kwargs={"k": top_k})

        # 3. The Ensemble (Reciprocal Rank Fusion)
        # This merges the two lists. We weight Chroma slightly higher (0.6)
        # because users usually ask natural language questions.
        print("   -> Calibrating Reciprocal Rank Fusion Algorithm...")
        self.hybrid_retriever = EnsembleRetriever(
            retrievers=[self.bm25_retriever, self.vector_retriever],
            weights=[0.4, 0.6]
        )
        print("[INFO] Hybrid Search is live and ready.")

    def get_retriever(self):
        """Returns the fully fused search interface."""
        return self.hybrid_retriever



## Testing database

In [ ]:
db_manager = VectorDBManager(persist_directory="./my_secure_rag_db")

# 1. Upsert the chunks we made in the last step
db_manager.upsert_chunks(semantic_results)

# 2. Test the retrieval to prove the database works
print("\n Testing Search Engine...")
retriever = db_manager.get_retriever(top_k=1)

# Let's search for something specific we know is in the text
test_query = "How did the Romans fight?"
search_results = retriever.invoke(test_query)

if search_results:
    best_match = search_results[0] # zero_bracket search_results
    print(f"Query: '{test_query}'")
    print(f"Top Result: '{best_match.page_content}'")
    print(f"Source: {best_match.metadata}")


# --- Execution ---
# Let's upgrade the database we built in the previous step
# (Assuming db_manager.vector_store and final_unique_docs are in memory)

hybrid_manager = HybridRetrieverManager(
    vector_store=db_manager.vector_store,
    documents=final_unique_docs,
    top_k=3
)

# Grab the upgraded search engine
production_retriever = hybrid_manager.get_retriever()

🔌 Initializing Google Gemini Embedding Engine (text-embedding-004)...
🗄️ Connecting to local Chroma Database (Dir: ./my_secure_rag_db)...
📥 Generating vectors and upserting 2 chunks into Chroma...
✅ Upsert complete! Your data is now mathematically searchable.

🔍 Testing Search Engine...
Query: 'How did the Romans fight?'
Top Result: 'Rome's military legions were highly disciplined. In completely unrelated news, quantum computing uses qubits to process information in ways classical computers cannot. By leveraging superposition, they solve complex math instantly.'
Source: {'source': 'test_comparison.txt'}
⚙️ Initializing Hybrid Retrieval Engine...
   -> Building BM25 Sparse Keyword Index...
   -> Connecting Chroma Dense Vector Index...
   -> Calibrating Reciprocal Rank Fusion Algorithm...
✅ Hybrid Search is live and ready.


# CORE

In [ ]:
def build_knowledge_base(
    data_dir: str = "./data",
    urls: list[str] = None,
    chunk_mode: str = "recursive",   # "recursive" or "semantic"
    chunk_size: int = 1000,
    chunk_overlap: int = 150,
    semantic_percentile: int = 80,
    collection_name: str = "rag_production",
    persist_directory: str = "./chroma_db",
    top_k: int = 4,
):
    """
    End-to-end ingestion pipeline: load -> clean -> dedupe -> chunk -> embed -> store.
    Returns (hybrid_retriever, vector_db_manager, final_chunks).
    """

    # 1. LOAD
    print("=" * 60)
    print("STEP 1: LOADING")
    print("=" * 60)
    raw_docs = []
    if data_dir is not None and Path(data_dir).exists():
        raw_docs.extend(load_from_directory(data_dir))
    else:
        print(f"[WARNING] Directory '{data_dir}' not found — skipping local files.")

    if urls:
        raw_docs.extend(load_from_urls(urls))

    if not raw_docs:
        raise ValueError(
            "[ERROR] No documents were loaded. Check DATA_DIR / urls and make sure "
            "the files actually exist in DATA_DIR."
        )

    # 2. CLEAN
    print("\n" + "=" * 60)
    print("STEP 2: CLEANING")
    print("=" * 60)
    cleaned_docs = process_and_clean_docs(raw_docs)

    # 3. DEDUPE
    print("\n" + "=" * 60)
    print("STEP 3: DEDUPLICATION")
    print("=" * 60)
    unique_docs = deduplicate_docs(cleaned_docs)

    # 4. CHUNK
    print("\n" + "=" * 60)
    print("STEP 4: CHUNKING")
    print("=" * 60)
    if chunk_mode == "semantic":
        embeddings_for_chunking = get_embedding_model()
        chunker = my_SemanticChunker(embedding_model=embeddings_for_chunking, percentile=semantic_percentile)
    else:
        chunker = my_RecursiveChunker(chunk_size=chunk_size, chunk_overlap=chunk_overlap)

    final_chunks = chunker.split(unique_docs)

    # 5. EMBED + STORE
    print("\n" + "=" * 60)
    print("STEP 5: VECTOR STORE")
    print("=" * 60)
    db_manager = VectorDBManager(collection_name=collection_name, persist_directory=persist_directory)
    db_manager.upsert_chunks(final_chunks)

    # 6. HYBRID RETRIEVER (vector + BM25)
    print("\n" + "=" * 60)
    print("STEP 6: HYBRID RETRIEVER")
    print("=" * 60)
    hybrid_manager = HybridRetrieverManager(
        vector_store=db_manager.vector_store,
        documents=final_chunks,
        top_k=top_k,
    )

    print("\n[INFO] Knowledge base ready.")
    return hybrid_manager.get_retriever(), db_manager, final_chunks


In [ ]:
from langchain_core.caches import InMemoryCache
from langchain_community.cache import SQLiteCache
from langchain_core.globals import set_llm_cache

def enable_llm_cache(persist: bool = True, db_path: str = "./llm_cache.db"):
    """
    persist=True  -> SQLiteCache, survives notebook restarts, best while iterating on the same questions.
    persist=False -> InMemoryCache, wiped on runtime reset, fine for a single session.
    """
    if persist:
        set_llm_cache(SQLiteCache(database_path=db_path))
        print(f"[INFO] LLM cache enabled (persistent, SQLite: {db_path})")
    else:
        set_llm_cache(InMemoryCache())
        print("[INFO] LLM cache enabled (in-memory, cleared on restart)")


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

RAG_PROMPT = ChatPromptTemplate.from_template(
    """You are a helpful assistant answering questions using ONLY the context below.
If the answer isn't in the context, say you don't know — do not make anything up.
Cite the source_file for any fact you use.

Context:
{context}

Question: {question}

Answer:"""
)


def format_docs(docs: list[Document], max_context_chars: int = 6000) -> str:
    """
    Turn retrieved chunks into a single context string, tagged by source —
    stopping once max_context_chars is reached so one query can't silently
    stuff a huge (expensive) prompt.
    """
    blocks = []
    total_chars = 0
    for d in docs:
        source = d.metadata.get("source_file", d.metadata.get("source", "unknown"))
        block = f"[Source: {source}]\n{d.page_content}"
        if total_chars + len(block) > max_context_chars:
            print(f"[WARNING] Context budget ({max_context_chars} chars) reached — "
                  f"dropped {len(docs) - len(blocks)} lower-ranked chunk(s).")
            break
        blocks.append(block)
        total_chars += len(block)
    return "\n\n---\n\n".join(blocks)


def build_rag_chain(retriever, model_name: str = "gemini-2.5-flash", temperature: float = 0.0,
                     max_output_tokens: int = 1024, max_context_chars: int = 6000):
    """
    Wire retriever -> prompt -> LLM -> parsed string into one runnable chain.

    max_output_tokens: hard cap on how long a single answer can be (output-side cost).
    max_context_chars: hard cap on how much retrieved text goes into the prompt (input-side cost,
                        usually the bigger cost driver in RAG since it's resent on every call).
    """
    llm = ChatGoogleGenerativeAI(model=model_name, temperature=temperature, max_output_tokens=max_output_tokens)

    def format_with_budget(docs):
        return format_docs(docs, max_context_chars=max_context_chars)

    chain = (
        {"context": retriever | format_with_budget, "question": RunnablePassthrough()}
        | RAG_PROMPT
        | llm
        | StrOutputParser()
    )
    return chain


In [ ]:
import time

# Rough threshold to tell a cache hit from a real API call. A real Gemini call
# normally takes ~1-4s; a SQLite cache read takes a few ms. This is a heuristic
# based on timing, NOT an official "was_cached" flag from LangChain -- there
# isn't one exposed on the chain output.
CACHE_LATENCY_THRESHOLD_SECONDS = 0.5

def ask(chain, retriever, question: str, show_sources: bool = True):
    """Ask a question and print the answer, latency, likely cache status, and sources used."""
    start = time.perf_counter()
    answer = chain.invoke(question)
    elapsed = time.perf_counter() - start

    likely_cached = elapsed < CACHE_LATENCY_THRESHOLD_SECONDS
    status = "️ CACHE" if likely_cached else " LIVE LLM CALL"

    print(f" Question: {question}\n")
    print(f" Answer:\n{answer}\n")
    print(f"️ Latency: {elapsed:.3f}s  |  Source: {status}\n")

    if show_sources:
        docs = retriever.invoke(question)
        print(" Sources used:")
        for d in docs:
            source = d.metadata.get("source_file", d.metadata.get("source", "unknown"))
            print(f"  - {source}: '{d.page_content[:100]}...'")
    return answer


## Testing

In [ ]:
import os
from pathlib import Path

# 1. Create the data directory if it doesn't exist
DATA_DIR = "./data"
os.makedirs(DATA_DIR, exist_ok=True)

# 2. Check environment (Colab upload vs Local directory)
try:
    from google.colab import files
    print(" Click the button below to upload your RAG documents:")
    uploaded = files.upload()

    # Move the uploaded files into data directory
    for filename in uploaded.keys():
        old_path = filename
        new_path = os.path.join(DATA_DIR, filename)
        os.rename(old_path, new_path)
        print(f"[INFO] Moved '{filename}' to {DATA_DIR}/")
except ImportError:
    # Local environment
    existing_files = [f for f in Path(DATA_DIR).glob("*") if f.is_file()]
    print(f" Local data directory: '{os.path.abspath(DATA_DIR)}'")
    if existing_files:
        print(f"[INFO] Found {len(existing_files)} file(s) in {DATA_DIR}:")
        for f in existing_files:
            print(f"   - {f.name}")
    else:
        print(f"[INFO] Place your PDF, DOCX, TXT, MD, or CSV documents into '{DATA_DIR}' to index them.")

👇 Click the button below to upload your RAG documents:


In [ ]:
# Example run — change DATA_DIR to wherever your files live locally or in this session
DATA_DIR = "./data"   # <-- put your PDFs / docx / txt / md / csv here

production_retriever, db_manager, final_chunks = build_knowledge_base(
    data_dir=DATA_DIR,
    chunk_mode="recursive",     # switch to "semantic" for topic-aware chunking (slower, uses embedding calls)
    chunk_size=1000,
    chunk_overlap=150,
    persist_directory="./chroma_db",
    top_k=4,
)


STEP 1: LOADING
📂 Found 1 files to load
  ✓ DOA_System_Brief.pdf: 3 doc(s)
✅ Loaded 3 total documents

STEP 2: CLEANING

STEP 3: DEDUPLICATION
🧹 Deduped 3 → 3 documents

STEP 4: CHUNKING

🔪 Executing RECURSIVE Chunking on 3 document(s)...
✅ Generated 8 chunks.

STEP 5: VECTOR STORE
🔌 Initializing Google Gemini Embedding Engine (text-embedding-004)...


/tmp/ipykernel_1043/3541287859.py:27: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "ingested_at": datetime.utcnow().isoformat(),


🗄️ Connecting to local Chroma Database (Dir: ./chroma_db)...
📥 Generating vectors and upserting 8 chunks into Chroma...
✅ Upsert complete! Your data is now mathematically searchable.

STEP 6: HYBRID RETRIEVER
⚙️ Initializing Hybrid Retrieval Engine...
   -> Building BM25 Sparse Keyword Index...
   -> Connecting Chroma Dense Vector Index...
   -> Calibrating Reciprocal Rank Fusion Algorithm...
✅ Hybrid Search is live and ready.

✅ Knowledge base ready.


In [ ]:
# Turn on the LLM cache BEFORE any calls, so repeated questions can actually hit it
enable_llm_cache(persist=True)

# Build the chain on top of the hybrid retriever from the ingestion step
rag_chain = build_rag_chain(production_retriever)

# Try it out — same question twice: first call hits the LLM, second should hit the cache
_ = ask(rag_chain, production_retriever, "What does this pdf talk about?")
_ = ask(rag_chain, production_retriever, "What does this pdf talk about?")


✅ LLM cache enabled (persistent, SQLite: ./llm_cache.db)
❓ Question: What does this pdf talk about?

💬 Answer:
The PDF talks about an FPGA-Based Real-Time Direction of Arrival (DOA) Estimation System. (DOA_System_Brief.pdf) This system is designed to identify the spatial origin of up to two simultaneous radio or acoustic signals in real time. (DOA_System_Brief.pdf) It takes raw antenna samples from a four-element array as input and produces bearing angle estimates as output. (DOA_System_Brief.pdf) The core algorithm used is MUSIC (Multiple Signal Classification). (DOA_System_Brief.pdf) The system determines the angle of each source, scanning a field of view from −60° to +60° in steps of 0.1°. (DOA_System_Brief.pdf)

⏱️ Latency: 4.219s  |  Source: 🌐 LIVE LLM CALL (likely)

📚 Sources used:
  - DOA_System_Brief.pdf: 'and electronic intelligence applications. The design has been architected as a streaming pipeline, w...'
  - DOA_System_Brief.pdf: 'FPGA-Based Real-Time DOA Estimation System

/usr/local/lib/python3.12/dist-packages/langchain_community/cache.py:265: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(row[0]) for row in rows]


❓ Question: What does this pdf talk about?

💬 Answer:
The PDF talks about an FPGA-Based Real-Time Direction of Arrival (DOA) Estimation System. (DOA_System_Brief.pdf) This system is designed to identify the spatial origin of up to two simultaneous radio or acoustic signals in real time. (DOA_System_Brief.pdf) It takes raw antenna samples from a four-element array as input and produces bearing angle estimates as output. (DOA_System_Brief.pdf) The core algorithm used is MUSIC (Multiple Signal Classification). (DOA_System_Brief.pdf) The system determines the angle of each source, scanning a field of view from −60° to +60° in steps of 0.1°. (DOA_System_Brief.pdf)

⏱️ Latency: 0.351s  |  Source: 🗄️ CACHE (likely)

📚 Sources used:
  - DOA_System_Brief.pdf: 'and electronic intelligence applications. The design has been architected as a streaming pipeline, w...'
  - DOA_System_Brief.pdf: 'FPGA-Based Real-Time DOA Estimation System Technical Brief · March 2026 Confidential — For Client Re...'
 